# Orbital Performance Report

Enable **Write performance CSV** in the debug UI's Performance panel, then copy the resulting file here or set `ORBITAL_PERF_CSV` to its path.

The notebook loads `performance_meta.json` next to the CSV when available (or `ORBITAL_PERF_META`) as a device-defaults snapshot. It then computes summary statistics and automated observations, and plots FPS, frame time, render resolution and scaling, distributions, CPU/GPU stages, and cache counters.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

CSV_PATH = Path(os.environ.get("ORBITAL_PERF_CSV", "Logs/performance.csv")).expanduser()
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Copy performance.csv here or set ORBITAL_PERF_CSV."
    )

df = pd.read_csv(CSV_PATH).sort_values("elapsed_s").reset_index(drop=True)
df.head()

In [ ]:
META_PATH = Path(
    os.environ.get("ORBITAL_PERF_META", CSV_PATH.with_name("performance_meta.json"))
).expanduser()

metadata = None
if META_PATH.exists():
    metadata = json.loads(META_PATH.read_text())
    print(f"Loaded capture metadata from {META_PATH}")
else:
    print(f"No metadata sidecar at {META_PATH}; device defaults unavailable.")

def meta(path, default=None):
    node = metadata or {}
    for key in path.split("."):
        if not isinstance(node, dict) or key not in node:
            return default
        node = node[key]
    return node

device_rows = {
    "app": f"{meta('app.name', '?')} {meta('app.version', '')}".strip(),
    "os / arch": f"{meta('app.os', '?')} / {meta('app.arch', '?')}",
    "gpu": meta("gpu.name", "?"),
    "driver": meta("gpu.driver", "?"),
    "window": f"{meta('window.width', '?')}x{meta('window.height', '?')} @ {meta('window.scale_factor', '?')}x",
    "surface": f"{meta('surface.width', '?')}x{meta('surface.height', '?')}",
    "present / refresh": f"{meta('surface.present_mode', '?')} / {meta('surface.refresh_rate_mhz', '?')} mHz",
    "target FPS": meta("render.target_fps", "?"),
    "render scale range": f"{meta('render.min_scale', '?')}..{meta('render.max_scale', '?')}x",
    "renderer": f"{meta('renderer.path', '?')}, {meta('renderer.anti_aliasing', '?')}",
    "scene": f"{meta('scene.entities', '?')} entities, {meta('scene.models', '?')} models, {meta('scene.lights', '?')} lights",
}
pd.Series(device_rows, name="value").to_frame()

In [ ]:
CACHE_COUNTERS = [
    "shader_modules_delta",
    "pipelines_delta",
    "material_hits_delta",
    "material_misses_delta",
]
SKY_COUNTER = "sky_updates_delta"
COUNTERS = [*CACHE_COUNTERS, "unique_wgsl", SKY_COUNTER]
SURFACE_COUNTERS = [
    "surface_success_total",
    "surface_suboptimal_total",
    "surface_timeout_total",
    "surface_occluded_total",
    "surface_outdated_total",
    "surface_lost_total",
    "surface_validation_total",
]
CAPTURE_FIELDS = [
    "window_width",
    "window_height",
    "surface_width",
    "surface_height",
    "render_width",
    "render_height",
    "render_scale",
    "scale_factor",
    "dpi",
    "orientation",
    "refresh_rate_mhz",
    "present_mode",
    "frame_latency",
    "fps_limit",
    "pixels_per_frame",
    "render_scale_changes",
    "render_resolution_changes",
    "gpu_timestamps_supported",
    "gpu_timestamps_stale",
]
TIMING_FIELDS = [
    "frames",
    "cdt_mean_ms",
    "cdt_min_ms",
    "cdt_max_ms",
    "cdt_p95_ms",
]
DISTRIBUTION_SUFFIXES = ("_min", "_max", "_p95")
DISTRIBUTION_COLUMNS = [
    column
    for column in df.columns
    if any(column.endswith(suffix) for suffix in DISTRIBUTION_SUFFIXES)
]
NON_STAGE_COLUMNS = {
    "elapsed_s",
    "fps",
    "cdt_ms",
    *COUNTERS,
    *SURFACE_COUNTERS,
    "visible_overlays",
    "visible_overlay_names",
    *CAPTURE_FIELDS,
    *TIMING_FIELDS,
    *DISTRIBUTION_COLUMNS,
}
STAGES = [c for c in df.columns if c not in NON_STAGE_COLUMNS]
CPU_STAGES = [c for c in STAGES if not c.startswith("gpu_") and c != "total"]
GPU_STAGES = [c for c in STAGES if c.startswith("gpu_")]
TOTAL = df["cdt_ms"] if "cdt_ms" in df else df["total"]
HAS_RENDER = all(name in df.columns for name in ("render_width", "render_height", "render_scale", "pixels_per_frame"))
HAS_DISTRIBUTIONS = bool(DISTRIBUTION_COLUMNS)

if HAS_RENDER:
    CHANGE_MASK = (
        (df.get("render_scale_changes", pd.Series(0, index=df.index)).fillna(0) > 0)
        | (df.get("render_resolution_changes", pd.Series(0, index=df.index)).fillna(0) > 0)
    )
else:
    CHANGE_MASK = pd.Series(False, index=df.index)

print(f"{len(df)} samples over {df['elapsed_s'].max() - df['elapsed_s'].min():.1f}s")
print(f"CPU stages: {', '.join(CPU_STAGES)}")
print(f"GPU stages: {', '.join(GPU_STAGES) if GPU_STAGES else 'none'}")
print("Capture fields:", ", ".join(f for f in CAPTURE_FIELDS if f in df))
if HAS_DISTRIBUTIONS:
    print(f"Stage distribution columns: {len(DISTRIBUTION_COLUMNS)}")
if HAS_RENDER:
    print(f"Render scale range: {df['render_scale'].min():.2f}x..{df['render_scale'].max():.2f}x")
    print(f"Rows with a scale or resolution change: {int(CHANGE_MASK.sum())}")

## Summary

`fps` is the one-second frame count. `total` is the renderer total; `cdt_ms` is the wall-clock delta between frames. `update + surface_acq + total + frame_limiter + frame_gap` approximates the full frame period.

In [ ]:
def pct(series, q):
    return float(np.nanpercentile(series.astype(float), q))

fps = df["fps"].astype(float)
cdt = df["cdt_ms"].astype(float)
means = df[STAGES].mean(numeric_only=True).sort_values(ascending=False)

summary_data = {
    "samples": len(df),
    "duration_s": float(df["elapsed_s"].max() - df["elapsed_s"].min()),
    "fps_mean": float(fps.mean()),
    "fps_median": float(fps.median()),
    "fps_p95": pct(fps, 95),
    "fps_p99": pct(fps, 99),
    "frame_total_mean_ms": float(TOTAL.mean()),
    "frame_total_p95_ms": pct(TOTAL, 95),
    "frame_total_p99_ms": pct(TOTAL, 99),
    "cdt_mean_ms": float(cdt.mean()),
    "cdt_p95_ms": pct(cdt, 95),
    "slowest_stage": str(means.index[0]) if len(means) else "-",
    "slowest_stage_mean_ms": float(means.iloc[0]) if len(means) else 0.0,
}

if "frames" in df:
    summary_data["frames_mean"] = float(df["frames"].astype(float).mean())
if "cdt_p95_ms" in df:
    summary_data["cdt_p95_ms"] = float(df["cdt_p95_ms"].astype(float).mean())
if "gpu_timestamps_stale" in df:
    summary_data["gpu_timestamp_stale_rows"] = int(df["gpu_timestamps_stale"].astype(bool).sum())

if HAS_RENDER:
    scale = df["render_scale"].astype(float)
    summary_data.update({
        "render_scale_mean": float(scale.mean()),
        "render_scale_min": float(scale.min()),
        "render_scale_max": float(scale.max()),
        "render_pixels_mean_mp": float(df["pixels_per_frame"].astype(float).mean() / 1e6),
        "render_scale_changes": int(df.get("render_scale_changes", pd.Series(0, index=df.index)).fillna(0).sum()),
        "render_resolution_changes": int(df.get("render_resolution_changes", pd.Series(0, index=df.index)).fillna(0).sum()),
    })

target_fps = meta("render.target_fps") if metadata else None
if target_fps:
    summary_data["target_fps"] = target_fps
    summary_data["fps_vs_target_pct"] = float(fps.mean() / float(target_fps) * 100.0)

pd.Series(summary_data).to_frame("value")

## Render scale and resolution

`render_scale`, `render_width`, `render_height`, and `pixels_per_frame` are recorded per timing row. `render_scale_changes` and `render_resolution_changes` mark rows where adaptive scaling, a manual scale change, a rotation, or a surface resize changed the internal target. A non-zero counter means that row covers a transition; exclude those rows when comparing steady-state frame rates.

The timeline below overlays FPS with the internal scale and pixel count. Vertical markers show change rows.

In [ ]:
if HAS_RENDER:
    scale = df["render_scale"].astype(float)
    pixels_mp = df["pixels_per_frame"].astype(float) / 1e6
    target_fps = meta("render.target_fps") if metadata else None

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    axes[0].plot(df["elapsed_s"], fps, color="#4aa3ff", linewidth=1.2)
    if target_fps:
        axes[0].axhline(float(target_fps), color="#55c98a", linestyle="--", linewidth=1, label=f"target {target_fps} FPS")
        axes[0].legend()
    axes[0].set_title("FPS and internal render resolution")
    axes[0].set_ylabel("frames/s")
    axes[0].grid(alpha=0.25)

    axes[1].plot(df["elapsed_s"], scale, color="#d16bff", linewidth=1.2)
    axes[1].set_ylabel("render scale")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].grid(alpha=0.25)

    axes[2].plot(df["elapsed_s"], pixels_mp, color="#ffb347", linewidth=1.2)
    axes[2].set_ylabel("render pixels (MPx)")
    axes[2].set_xlabel("elapsed_s")
    axes[2].grid(alpha=0.25)

    for timestamp in df.loc[CHANGE_MASK, "elapsed_s"]:
        for ax in axes:
            ax.axvline(timestamp, color="#ff4f4f", linestyle=":", linewidth=0.8, alpha=0.7)

    plt.tight_layout()
    plt.show()
else:
    print("No render-resolution columns found; rerun with a current Orbital build.")

In [ ]:
if HAS_RENDER:
    change_rows = df.loc[
        CHANGE_MASK,
        [
            "elapsed_s",
            "fps",
            "render_width",
            "render_height",
            "render_scale",
            "pixels_per_frame",
            "render_scale_changes",
            "render_resolution_changes",
        ],
    ].copy()
    change_rows["render_pixels_mp"] = change_rows["pixels_per_frame"] / 1e6
    print(f"Rows covering a render-scale or resolution change: {len(change_rows)}")
    display(change_rows)

    work = df.assign(
        render_key=(
            df["render_width"].astype(int).astype(str)
            + "x"
            + df["render_height"].astype(int).astype(str)
            + " @ "
            + df["render_scale"].map(lambda value: f"{value:.2f}x")
        ),
        render_pixels_mp=df["pixels_per_frame"].astype(float) / 1e6,
        frame_ms=TOTAL.astype(float),
    )
    segments = (
        work.groupby("render_key", dropna=False)
        .agg(
            samples=("elapsed_s", "size"),
            duration_s=("elapsed_s", lambda values: float(values.max() - values.min())),
            fps_mean=("fps", "mean"),
            fps_median=("fps", "median"),
            frame_ms_mean=("frame_ms", "mean"),
            render_pixels_mp=("render_pixels_mp", "mean"),
        )
        .reset_index()
        .sort_values("render_pixels_mp", ascending=False)
    )
    segments["sample_share_pct"] = segments["samples"] / len(df) * 100.0
    display(segments)
else:
    print("No render-resolution columns found; rerun with a current Orbital build.")

In [ ]:
if HAS_RENDER:
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(
        df["pixels_per_frame"].astype(float) / 1e6,
        fps,
        c=df["render_scale"].astype(float),
        cmap="viridis",
        s=28,
        alpha=0.85,
    )
    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label("render scale")

    target_fps = meta("render.target_fps") if metadata else None
    if target_fps:
        ax.axhline(float(target_fps), color="#55c98a", linestyle="--", linewidth=1, label=f"target {target_fps} FPS")
        ax.legend()

    ax.set_title("FPS vs internal render resolution")
    ax.set_xlabel("render pixels (MPx)")
    ax.set_ylabel("FPS")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("No render-resolution columns found; rerun with a current Orbital build.")

In [ ]:
spike_threshold = max(float(TOTAL.median()) * 2.0, 1e-6)
spikes = int((TOTAL > spike_threshold).sum())
fps_cv = float(fps.std() / fps.mean() * 100.0) if fps.mean() else 0.0

print(f"Frame-time spikes above {spike_threshold:.2f} ms: {spikes}")
print(f"FPS coefficient of variation: {fps_cv:.1f}%")
print("\nMean stage cost:")
for name, value in means.head(12).items():
    share = value / TOTAL.mean() * 100.0 if TOTAL.mean() else 0.0
    print(f"  {name:24s} {value:8.3f} ms  ({share:5.1f}%)")

if fps_cv > 10.0:
    print("\nFPS is noticeably variable; inspect the frame-time and stage graphs.")
if spikes:
    print("Spikes are present; compare their timestamps with stage peaks.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df["elapsed_s"], fps, color="#4aa3ff", linewidth=1.2)
axes[0].set_title("FPS")
axes[0].set_ylabel("frames/s")
axes[0].grid(alpha=0.25)

axes[1].plot(df["elapsed_s"], TOTAL, label="frame total", color="#ffb347", linewidth=1.1)
axes[1].plot(df["elapsed_s"], cdt, label="CDT", color="#d16bff", linewidth=0.9)
axes[1].set_title("Frame time")
axes[1].set_ylabel("ms")
axes[1].set_xlabel("elapsed_s")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(fps, bins=30, color="#4aa3ff", alpha=0.8)
axes[0].axvline(fps.median(), color="#ffb347", linestyle="--", label="median")
axes[0].set_title("FPS distribution")
axes[0].set_xlabel("frames/s")
axes[0].legend()

axes[1].hist(TOTAL, bins=40, color="#ff8c69", alpha=0.8)
axes[1].axvline(TOTAL.median(), color="#ffb347", linestyle="--", label="median")
axes[1].set_title("Frame-time distribution")
axes[1].set_xlabel("ms")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
if CPU_STAGES:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    cpu_means = df[CPU_STAGES].mean(numeric_only=True).sort_values()
    axes[0].barh(cpu_means.index, cpu_means.values, color="#55c98a")
    axes[0].set_title("CPU stage mean")
    axes[0].set_xlabel("ms")

    top = cpu_means.sort_values(ascending=False).head(6).index
    for name in top:
        axes[1].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[1].set_title("Top CPU stages over time")
    axes[1].set_xlabel("elapsed_s")
    axes[1].set_ylabel("ms")
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("No CPU stage columns found.")

In [ ]:
if CPU_STAGES:
    top_cpu = df[CPU_STAGES].mean(numeric_only=True).sort_values(ascending=False).head(10).index
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].stackplot(
        df["elapsed_s"],
        *[df[name] for name in top_cpu],
        labels=top_cpu,
        alpha=0.85,
    )
    axes[0].set_title("CPU composition (top 10 stages)")
    axes[0].set_xlabel("elapsed_s")
    axes[0].set_ylabel("ms")
    cpu_means = df[CPU_STAGES].mean(numeric_only=True)
    shares = (cpu_means.loc[top_cpu] / TOTAL.mean() * 100.0).sort_values()
    axes[1].barh(shares.index, shares.values, color="#79d6a6")
    axes[1].set_title("CPU share of mean frame time")
    axes[1].set_xlabel("%")
    plt.tight_layout()
    plt.show()
else:
    print("No CPU stage columns found.")

In [ ]:
if GPU_STAGES:
    gpu_means = df[GPU_STAGES].mean(numeric_only=True).sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for name in GPU_STAGES:
        axes[0].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[0].set_title("GPU stages over time")
    axes[0].set_xlabel("elapsed_s")
    axes[0].set_ylabel("ms")
    axes[0].legend(fontsize=7, ncol=2)
    axes[0].grid(alpha=0.25)

    axes[1].barh(gpu_means.index, gpu_means.values, color="#ff8c69")
    axes[1].set_title("GPU stage mean")
    axes[1].set_xlabel("ms")
    plt.tight_layout()
    plt.show()
else:
    print("No GPU stage columns found; timestamp queries may be unavailable.")

## Timing distributions

The CSV keeps the per-stage mean columns for backward compatibility and adds `<stage>_min`, `<stage>_max`, and `<stage>_p95` columns for every stage in the same window. The table below ranks stages by mean p95, so short hitch spikes are visible even when the average looks healthy.

In [ ]:
if HAS_DISTRIBUTIONS:
    rows = []
    for column in DISTRIBUTION_COLUMNS:
        if not column.endswith("_p95"):
            continue
        stage = column[: -len("_p95")]
        if stage not in df:
            continue
        min_column = f"{stage}_min"
        max_column = f"{stage}_max"
        rows.append({
            "stage": stage,
            "mean_ms": float(df[stage].astype(float).mean()),
            "p95_mean_ms": float(df[column].astype(float).mean()),
            "p95_max_ms": float(df[column].astype(float).max()),
            "max_ms": float(df[max_column].astype(float).max()) if max_column in df else float("nan"),
            "min_ms": float(df[min_column].astype(float).min()) if min_column in df else float("nan"),
        })

    distribution = pd.DataFrame(rows).sort_values("p95_mean_ms", ascending=False)
    display(distribution.head(20))

    top = distribution.head(6)["stage"].tolist()
    if top:
        fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
        for stage in top:
            axes[0].plot(df["elapsed_s"], df[f"{stage}_p95"], label=f"{stage} p95", linewidth=0.9)
        axes[0].set_title("Top stage p95 over time")
        axes[0].set_xlabel("elapsed_s")
        axes[0].set_ylabel("ms")
        axes[0].legend(fontsize=7, ncol=2)
        axes[0].grid(alpha=0.25)

        ranked = distribution.set_index("stage")
        width = 0.25
        positions = np.arange(len(top))
        axes[1].bar(positions - width, ranked.loc[top, "mean_ms"], width, label="mean")
        axes[1].bar(positions, ranked.loc[top, "p95_mean_ms"], width, label="p95")
        axes[1].bar(positions + width, ranked.loc[top, "max_ms"], width, label="max")
        axes[1].set_xticks(positions, top, rotation=25, ha="right")
        axes[1].set_title("Mean / p95 / max by stage")
        axes[1].set_ylabel("ms")
        axes[1].legend(fontsize=8)
        axes[1].grid(alpha=0.25, axis="y")
        plt.tight_layout()
        plt.show()
else:
    print("No stage distribution columns found; rerun with a current Orbital build.")

In [ ]:
rolling_window = max(3, min(15, len(TOTAL) // 20 or 3))
rolling = TOTAL.rolling(window=rolling_window, min_periods=1).mean()
spike_mask = TOTAL > spike_threshold

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["elapsed_s"], TOTAL, label="frame total", color="#ffb347", alpha=0.5, linewidth=0.9)
ax.plot(df["elapsed_s"], rolling, label=f"rolling mean ({rolling_window})", color="#4aa3ff", linewidth=1.5)
ax.scatter(
    df.loc[spike_mask, "elapsed_s"],
    TOTAL[spike_mask],
    color="#ff4f4f",
    s=18,
    label=f"spikes ({spikes})",
    zorder=3,
)
ax.axhline(TOTAL.median(), color="#55c98a", linestyle="--", linewidth=1, label="median")
ax.set_title("Frame-time spikes and rolling mean")
ax.set_xlabel("elapsed_s")
ax.set_ylabel("ms")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
cache_present = [name for name in CACHE_COUNTERS if name in df]
if cache_present:
    for name in cache_present:
        axes[0].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[0].set_title("Cache activity (per window)")
    axes[0].legend(fontsize=7, ncol=2)
else:
    axes[0].set_visible(False)

if "unique_wgsl" in df:
    axes[1].plot(df["elapsed_s"], df["unique_wgsl"], color="#4aa3ff")
    axes[1].set_title("Unique WGSL sources (cumulative)")
else:
    axes[1].set_visible(False)

if SKY_COUNTER in df:
    axes[2].plot(df["elapsed_s"], df[SKY_COUNTER], color="#ffb347")
    axes[2].set_title("Dynamic sky updates (per window)")
    axes[2].set_ylabel("updates")
else:
    axes[2].set_visible(False)

for ax in axes:
    if ax.get_visible():
        ax.set_xlabel("elapsed_s")
        ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Reading the result

- Raw FPS is only directly comparable at matched surface resolution, internal render resolution, and refresh rate. Use the per-resolution segment table before comparing devices.
- `render_scale`, `render_width`, `render_height`, and `pixels_per_frame` describe the internal 3D target; 2D, UI, and text remain at surface resolution.
- `render_scale_changes` and `render_resolution_changes` mark transition rows. Exclude them when measuring steady-state performance, or inspect the change table to see how quickly the controller settled.
- Use the `<stage>_p95` and `<stage>_max` columns for hitch analysis. A healthy mean with a high p95 or max points at intermittent CPU/GPU work.
- `frames` is the number of samples in the row's window; `cdt_mean_ms`, `cdt_min_ms`, `cdt_max_ms`, and `cdt_p95_ms` describe the wall-clock frame delta. `cdt_ms` remains the latest frame delta for backward compatibility.
- `gpu_timestamps_stale` means GPU timestamps did not advance between rows; ignore GPU stage comparisons for those rows.
- A high `total` with a low `present` usually means CPU work before submission.
- `scene_encode`, `scene_submit`, `cull+extract`, or `bind+models` growing together points at scene complexity or CPU-side preparation.
- `gpu_*` growth points at the GPU pass; if they are all zero, timestamp queries are unavailable on that backend.
- `gpu_volumetrics`, `gpu_tonemap`, and `gpu_aa` cover post-processing GPU work; `gpu_scene_total` covers shadow/skybox/models only.
- `realize` and `realize_models` are usually one-time setup costs, but if they stay high they indicate models are being treated as dirty every frame.
- `surface_acq` is the total, split into `surface_lock`, `surface_acquire`, and `surface_view`. A high `surface_acquire` is often swapchain/GPU backpressure, not CPU computation.
- `material_misses_delta` or `pipelines_delta` growing after startup often means materials or shaders are being created at runtime.
- `sky_updates_delta` is the dynamic-sky update count, not a cache metric. A dynamic sky can legitimately update roughly once per frame; compare it with FPS rather than the cache counters.
- `unique_wgsl` is cumulative. The other cache counters are per-window deltas, so they are plotted on separate axes.
- Compare the mean and p99, not just the average: a good mean with a bad p99 usually means hitching or occasional compile/upload work.